## Exercise — Data Retention and Deletion Workflow

## HealthBridge AI: Automated Deletion Pipeline with Audit Logging

This notebook implements key logic for processing **GDPR Article 17** (right-to-be-forgotten) and **CPRA §1798.105** (right to delete) deletion requests, tracking model-data lineage, and generating compliance reports.

---

### Quick Reference: Key Concepts

**GDPR Right to Erasure (Art. 17)**: Data subjects can request deletion of their personal data. Organizations must comply within one month under Art. 12(3) (extendable by two further months for complex requests) unless exemptions apply.

**CPRA Right to Delete (Cal. Civ. Code §1798.105)**: California consumers can request deletion of personal information collected from them. Businesses must comply within 45 days (extendable by another 45 days), unless an exception applies — including compliance with legal obligations, fraud prevention, or completion of the transaction for which the information was collected. CPRA verification requirements are stricter than GDPR for sensitive categories (e.g., government IDs, biometric data).

**Exceptions to Deletion** (both GDPR Art. 17(3) and CPRA §1798.105(d)):
- Legal obligations (compliance records, regulatory requirements such as FDA / clinical-trial holds, financial-record retention)
- Public health purposes (safety monitoring data)
- Archiving in the public interest (research data if properly anonymized)
- Defense of legal claims

**Model-Data Lineage**: Maps which training data feeds into which models, enabling impact assessment before deletion and retraining decisions after deletion.

**Deletion Methods**:
- **Secure Erase**: DoD 5220.22-M standard — for sensitive data on physical storage
- **Cryptographic Erasure**: Destroy encryption keys — for cloud/encrypted storage
- **Standard Delete**: For non-sensitive operational data
- **Archival**: Move to long-term storage after retention period (for audit logs)

**Retraining Decision Framework**:
- Critical/High impact model + in production + training data deleted → **Must retrain (HIGH priority)**
- Medium impact model + in production + training data deleted → **Should retrain (MEDIUM priority)**
- Low impact or archived model → **No immediate retraining needed**

### Scenario
HealthBridge AI manages patient data for a clinical notes summarization GenAI system. A patient (SUBJ-10001) has submitted a right-to-be-forgotten request. You need to implement the deletion logic, assess model impact, and generate compliance documentation.

> **Note on retention conflicts:** Some patient records are subject to **a regulatory hold** (e.g., clinical trial data with a 25-year retention requirement). Your `check_retention_conflict()` implementation must detect those and route them to a `legal_hold` state rather than deleting them, even when a valid right-to-erasure request comes in. This is the GDPR Art. 17(3)(b) / CPRA §1798.105(d)(7) exemption in action.


## Step 1: Setup and Data

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Tuple
import uuid
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 60)

# ============================================================
# DATA INVENTORY — All personal data records in HealthBridge AI
# Every record carries a `retention_requirement` field. Values are one of:
#   - 'GDPR-erasable'            (standard PII — eligible for Art. 17 deletion)
#   - 'Regulatory Hold - 25 years'      (clinical trial data — FDA hold under 21 CFR §312.62; the 25-year figure comes from EU CTR 536/2014 Art. 58)
#   - 'Consent record - keep'    (consent proof — must retain per GDPR Art. 7(1))
#   - 'Audit log - 6 years'      (HIPAA / SOX-style audit retention)
# Your check_retention_conflict() must read this field.
# ============================================================
data_inventory = pd.DataFrame([
    {
        'data_id': 'DEM-001-2023-01', 'subject_id': 'SUBJ-10001',
        'data_type': 'Patient Demographics', 'created_date': datetime(2023, 1, 15),
        'last_accessed': datetime(2025, 12, 10),
        'models_trained_on': ['Clinical Summary Model v2.1', 'Risk Stratification v3.0'],
        'storage_location': 'ehr_db.patients', 'sensitive': True, 'data_size_mb': 0.5,
        'data_usage': 'pre_training',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'CLIN-001-2023-02', 'subject_id': 'SUBJ-10001',
        'data_type': 'Clinical Notes', 'created_date': datetime(2023, 2, 20),
        'last_accessed': datetime(2025, 12, 5),
        'models_trained_on': ['Clinical Summary Model v2.1', 'Diagnosis Predictor v1.8'],
        'storage_location': 'ehr_db.clinical_notes', 'sensitive': True, 'data_size_mb': 2.1,
        'data_usage': 'pre_training',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'TRAIN-001-2023-03', 'subject_id': 'SUBJ-10001',
        'data_type': 'Model Training Dataset', 'created_date': datetime(2023, 3, 10),
        'last_accessed': datetime(2024, 6, 15),
        'models_trained_on': ['Clinical Summary Model v2.1', 'Diagnosis Predictor v1.8', 'Treatment Recommender v2.0'],
        'storage_location': 's3://training_data/subj-10001', 'sensitive': True, 'data_size_mb': 150.0,
        'data_usage': 'fine_tuning',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'INFER-001-2025-11', 'subject_id': 'SUBJ-10001',
        'data_type': 'Inference Logs', 'created_date': datetime(2025, 11, 1),
        'last_accessed': datetime(2026, 1, 15),
        'models_trained_on': ['Adverse Event Detector v3.2'],
        'storage_location': 'elasticsearch.inference_logs', 'sensitive': False, 'data_size_mb': 5.2,
        'data_usage': 'inference_log',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'QUERY-001-2025-10', 'subject_id': 'SUBJ-10001',
        'data_type': 'User Queries', 'created_date': datetime(2025, 10, 5),
        'last_accessed': datetime(2026, 1, 20),
        'models_trained_on': ['Query Intent Classifier v1.5'],
        'storage_location': 'postgres_db.user_queries', 'sensitive': False, 'data_size_mb': 0.8,
        'data_usage': 'prompt_history',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'CONS-001-2023-01', 'subject_id': 'SUBJ-10001',
        'data_type': 'Consent Records', 'created_date': datetime(2023, 1, 10),
        'last_accessed': datetime(2025, 12, 1),
        'models_trained_on': [],
        'storage_location': 'consent_db.consent_records', 'sensitive': True, 'data_size_mb': 0.2,
        'data_usage': 'inference_log',
        'retention_requirement': 'Consent record - keep',
    },
    {
        'data_id': 'TRIAL-001-2023-04', 'subject_id': 'SUBJ-10001',
        'data_type': 'Clinical Trial Results', 'created_date': datetime(2023, 4, 12),
        'last_accessed': datetime(2025, 11, 30),
        'models_trained_on': [],
        'storage_location': 'ehr_db.clinical_trials', 'sensitive': True, 'data_size_mb': 4.7,
        'data_usage': 'inference_log',
        'retention_requirement': 'Regulatory Hold - 25 years',
    },
    {
        'data_id': 'DEM-002-2023-05', 'subject_id': 'SUBJ-10002',
        'data_type': 'Patient Demographics', 'created_date': datetime(2023, 5, 12),
        'last_accessed': datetime(2025, 11, 20),
        'models_trained_on': ['Risk Stratification v3.0'],
        'storage_location': 'ehr_db.patients', 'sensitive': True, 'data_size_mb': 0.5,
        'data_usage': 'pre_training',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'CLIN-002-2023-06', 'subject_id': 'SUBJ-10002',
        'data_type': 'Clinical Notes', 'created_date': datetime(2023, 6, 1),
        'last_accessed': datetime(2025, 11, 15),
        'models_trained_on': ['Clinical Summary Model v2.1'],
        'storage_location': 'ehr_db.clinical_notes', 'sensitive': True, 'data_size_mb': 1.8,
        'data_usage': 'pre_training',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'data_id': 'BACKUP-001-2023-03', 'subject_id': 'SUBJ-10001',
        'data_type': 'Backup Data', 'created_date': datetime(2023, 3, 1),
        'last_accessed': datetime(2026, 2, 1),
        'models_trained_on': ['Clinical Summary Model v2.1', 'Diagnosis Predictor v1.8'],
        'storage_location': 's3://backups/full_backup_2023-03', 'sensitive': True, 'data_size_mb': 5000.0,
        'data_usage': 'pre_training',
        'retention_requirement': 'GDPR-erasable',
    },
    {
        'retention_requirement': 'GDPR-erasable',
        'data_id': 'RAG-CORPUS-001', 'subject_id': 'SUBJ-10001',
        'data_type': 'RAG Corpus', 'created_date': datetime(2025, 6, 1),
        'last_accessed': datetime(2026, 1, 20),
        'models_trained_on': ['Clinical Summary Model v2.1'],
        'storage_location': 'elasticsearch.rag_index', 'sensitive': True, 'data_size_mb': 12.0,
        'data_usage': 'rag_corpus'  # (L6-R002: rag_corpus usage)
    },
])

# ============================================================
# MODEL-DATA LINEAGE — Which models depend on which data types
# ============================================================
model_lineage = {
    'Clinical Summary Model v2.1': {
        'training_data_types': ['Patient Demographics', 'Clinical Notes', 'Model Training Dataset'],
        'deployment_date': datetime(2024, 3, 15),
        'status': 'in_production', 'impact_level': 'critical',
        'last_retrained': datetime(2025, 9, 20)
    },
    'Diagnosis Predictor v1.8': {
        'training_data_types': ['Clinical Notes', 'Model Training Dataset'],
        'deployment_date': datetime(2024, 6, 1),
        'status': 'in_production', 'impact_level': 'high',
        'last_retrained': datetime(2025, 11, 10)
    },
    'Risk Stratification v3.0': {
        'training_data_types': ['Patient Demographics', 'Clinical Notes'],
        'deployment_date': datetime(2024, 4, 20),
        'status': 'in_production', 'impact_level': 'high',
        'last_retrained': datetime(2025, 10, 5)
    },
    'Treatment Recommender v2.0': {
        'training_data_types': ['Clinical Notes', 'Model Training Dataset'],
        'deployment_date': datetime(2024, 8, 10),
        'status': 'in_production', 'impact_level': 'medium',
        'last_retrained': datetime(2025, 8, 15)
    },
    'Query Intent Classifier v1.5': {
        'training_data_types': ['User Queries'],
        'deployment_date': datetime(2024, 2, 1),
        'status': 'in_production', 'impact_level': 'low',
        'last_retrained': datetime(2025, 12, 1)
    },
    'Adverse Event Detector v3.2': {
        'training_data_types': ['Clinical Trial Results'],
        'deployment_date': datetime(2023, 12, 1),
        'status': 'in_production', 'impact_level': 'critical',
        'last_retrained': datetime(2025, 6, 1),
        'data_usage': 'pre_training'
    },
}

print(f'Data inventory: {len(data_inventory)} records across {data_inventory["subject_id"].nunique()} subjects')
print(f'Model lineage: {len(model_lineage)} models tracked')
print('\nData types in inventory:', data_inventory['data_type'].unique().tolist())
print('Retention requirements present:', sorted(data_inventory['retention_requirement'].unique().tolist()))
print('\nModels:')
for name, info in model_lineage.items():
    print(f'  {name}: {info["impact_level"]} impact, {info["status"]}')


## Step 2: Implement Deletion Method Selection


In [ ]:
def select_deletion_method(row: pd.Series) -> str:
    """
    Select the appropriate deletion method based on data sensitivity and storage type.
    
    Returns one of: 'secure_erase', 'cryptographic_erasure', 'standard_delete', 'archival'
    
    Decision logic:
    - Consent Records → 'archival' (must retain proof of consent per GDPR Art. 7)
    - Sensitive data on cloud storage (s3://) → 'cryptographic_erasure'
    - Sensitive data on database storage → 'secure_erase'
    - Non-sensitive data → 'standard_delete'
    
    NOTE (L6-R005): If deletion_method == 'archival', the status should be 
    'archived_with_access_restriction' instead of 'completed' during execution.
    """
    # TODO: Implement the deletion method selection logic
    # Hint: Check data_type first for special cases, then sensitive flag, then storage_location
    pass

## Step 3: Retention Conflict Detection (L6-R001: TODO)

Some records carry retention requirements that **override** a GDPR Article 17 deletion request — typically a regulatory hold (FDA clinical-trial Title 21 CFR §312.62, financial-records SOX retention, etc.). The deletion workflow needs to detect these and route them to a `legal_hold` state instead of executing the deletion.

Every record in `data_inventory` has a `retention_requirement` field. Your job: implement `check_retention_conflict(row)` that reads this field and returns a 2-tuple.


In [ ]:
# TODO Step 3: Implement check_retention_conflict(row).
#
# Spec:
#   - Accept a single inventory row (pd.Series) as input.
#   - Read row['retention_requirement']. Values you'll encounter:
#       'GDPR-erasable'          → no conflict, return ('deletable', '')
#       'Regulatory Hold - 25 years'    → regulatory hold; cannot delete
#       'Consent record - keep'  → must retain per GDPR Art. 7(1)
#       'Audit log - 6 years'    → if older than retention period, deletable; else hold
#   - For values OTHER than 'GDPR-erasable', return ('legal_hold', reason_string)
#     where reason_string names the regulatory anchor in one sentence.
#   - Return: 2-tuple (status, reason_string) where status is 'deletable' or 'legal_hold'.
#
# Why this exists: process_deletion_request will call this BEFORE selecting a
# deletion method. legal_hold rows are written to the audit trail as
# "EXEMPTED_FROM_DELETION" rather than being deleted.

def check_retention_conflict(row):
    """TODO: Implement legal-hold detection per the spec above."""
    return ('deletable', '')

print('check_retention_conflict() placeholder defined')


## Step 4: Flag Large Records (L6-R006: TODO)

For very large records (e.g., > 1 GB), deletion can't be a fire-and-forget operation — there are usually offsite backups on cold storage, backup-rotation schedules to update, and coordinator handoffs across storage systems. The deletion workflow needs to flag these records for **manual review** rather than processing them in the automated pipeline.

Your job: implement `flag_large_records(row)` to return a `{requires_manual_review, review_reason, notes}` dict.


In [ ]:
# TODO Step 4: Implement flag_large_records(row).
#
# Spec:
#   - Accept a single inventory row (pd.Series) as input.
#   - If row['data_size_mb'] exceeds SIZE_THRESHOLD_MB (start at 1000 MB = 1 GB),
#     set requires_manual_review=True and write a one-sentence review_reason
#     explaining WHY (large record + offsite backup + rotation considerations).
#   - Otherwise set requires_manual_review=False with empty review_reason.
#   - Return: {'requires_manual_review': bool, 'review_reason': str, 'notes': str}
#
# Why this exists: process_deletion_request will call this on every record. Rows
# that require manual review go to a queue rather than the automated deletion
# pipeline. The notes field is a free-text spot for "check tape backup #X" type
# coordinator instructions.

SIZE_THRESHOLD_MB = 1000  # 1 GB — adjust if your team's policy differs

def flag_large_records(row):
    """TODO: Implement large-record flagging per the spec above."""
    return {'requires_manual_review': False, 'review_reason': '', 'notes': ''}

print('flag_large_records() placeholder defined')


## Step 5: Implement Model Impact Assessment


In [ ]:
def assess_model_impact(deleted_data_types: List[str], model_lineage: dict) -> pd.DataFrame:
    """
    Assess which models are affected by data deletion and determine retraining priority.
    
    Returns DataFrame with columns: model_name, impact_level, status, overlap_data_types,
                                     retraining_required, priority
    
    Priority rules:
    - critical or high impact + in_production + data overlap → 'HIGH' priority, retraining 'YES'
    - medium impact + in_production + data overlap → 'MEDIUM' priority, retraining 'YES'
    - low impact or not in_production → 'LOW' priority, retraining 'NO'
    
    (L6-R002: Consider data_usage from model_lineage to assess fine-tuning vs. pre-training impact)
    """
    results = []
    
    for model_name, info in model_lineage.items():
        overlap = set(deleted_data_types) & set(info['training_data_types'])
        
        if not overlap:
            continue  # Model not affected by this deletion
        
        # TODO: Determine retraining_required ('YES' or 'NO') and priority ('HIGH', 'MEDIUM', 'LOW')
        # based on the model's impact_level and status
        retraining_required = 'TODO'
        priority = 'TODO'
        
        results.append({
            'model_name': model_name,
            'impact_level': info['impact_level'],
            'status': info['status'],
            'overlap_data_types': ', '.join(overlap),
            'last_retrained': info['last_retrained'],
            'days_since_retrain': (datetime.now() - info['last_retrained']).days,
            'retraining_required': retraining_required,
            'priority': priority
        })
    
    return pd.DataFrame(results).sort_values('priority') if results else pd.DataFrame()

## Step 6: Process Deletion Request


In [ ]:
def process_deletion_request(subject_id, data_inventory, model_lineage):
    """Process a GDPR Article 17 deletion request."""
    request_id = f'RTF-{uuid.uuid4().hex[:8].upper()}'
    report = {
        'request_id': request_id, 'subject_id': subject_id,
        'request_timestamp': datetime.now(), 'status': 'pending',
        'data_found': [], 'deletion_execution': [], 
        'affected_models': [], 'retraining_required': [],
        'total_data_size_mb': 0, 'audit_entries': [],
        'legal_holds': []
    }
    
    # Find all data for this subject
    subject_data = data_inventory[data_inventory['subject_id'] == subject_id].copy()
    
    if len(subject_data) == 0:
        report['status'] = 'no_data_found'
        report['audit_entries'].append({
            'timestamp': datetime.now(), 'event': 'NO_DATA_FOUND',
            'details': f'No records found for {subject_id}', 'status': 'completed'
        })
        return report
    
    report['data_found'] = subject_data[['data_id', 'data_type', 'storage_location', 'data_size_mb']].to_dict('records')
    report['total_data_size_mb'] = subject_data['data_size_mb'].sum()
    
    report['audit_entries'].append({
        'timestamp': datetime.now(), 'event': 'DATA_INVENTORY_CHECK',
        'details': f'Found {len(subject_data)} records ({report["total_data_size_mb"]:.1f} MB)',
        'status': 'completed'
    })
    
    # --- TODO #3: Execute deletion for each record (L6-R005: Check deletion method) ---
    # For each row in subject_data:
    #   1. Call select_deletion_method(row) to get the deletion method
    #   2. Create a deletion entry dict with: data_id, storage_location, deletion_method, 
    #      timestamp (use datetime.now()), status='completed', verification='verified_deleted'
    #   3. NOTE (L6-R005): If deletion_method == 'archival', set status='archived_with_access_restriction'
    #      and verification='archived' instead
    #   4. Append to report['deletion_execution']
    #   5. LEGAL HOLD (L6-R001): call check_retention_conflict(row) FIRST. If it returns
    #      ('legal_hold', reason), append {'data_id', 'data_type', 'reason'} to
    #      report['legal_holds'] and `continue` — do NOT delete that record.
    
    # YOUR CODE HERE (roughly 8-10 lines)
    pass
    
    _deleted  = sum(1 for e in report['deletion_execution'] if e['status'] == 'completed')
    _archived = sum(1 for e in report['deletion_execution'] if e['status'] == 'archived_with_access_restriction')
    _held     = len(report['legal_holds'])
    report['audit_entries'].append({
        'timestamp': datetime.now(), 'event': 'DELETION_EXECUTED',
        'details': (f'Deleted {_deleted} records; archived {_archived} under a retention obligation; '
                    f'{_held} retained under legal hold'),
        'status': 'completed'
    })
    
    # --- TODO #4: Assess model impact (L6-R002: Consider data_usage impact) ---
    # 1. Get the unique data_types that were deleted from subject_data
    # 2. Call assess_model_impact(deleted_data_types, model_lineage)
    # 3. Store the full DataFrame in report['affected_models']
    # 4. Filter for rows where retraining_required == 'YES' and store in report['retraining_required']
    # Hint (L6-R002): The assess_model_impact function now factors in data_usage type
    #      (pre_training requires full retrain, fine_tuning requires reset + re-finetune,
    #       rag_corpus requires index rebuild, prompt_history needs PII verification)
    
    # YOUR CODE HERE (roughly 4-6 lines)
    pass
    
    report['audit_entries'].append({
        'timestamp': datetime.now(), 'event': 'IMPACT_ASSESSMENT',
        'details': f'Model impact assessed; retraining flagged where needed',
        'status': 'completed'
    })
    
    report['status'] = 'completed'
    return report

print('process_deletion_request() defined')


## Step 7: Run Deletion Requests (3 scenarios)


In [ ]:
# Scenario 1: Patient with extensive data and model training dependencies
print('='*80)
print('DELETION REQUEST #1: SUBJ-10001 (Patient with clinical notes and training data)')
print('='*80)

report_1 = process_deletion_request('SUBJ-10001', data_inventory, model_lineage)

print(f'\nRequest ID: {report_1["request_id"]}')
print(f'Status: {report_1["status"]}')
print(f'Records found: {len(report_1["data_found"])}')
print(f'Total data: {report_1["total_data_size_mb"]:.1f} MB')

print(f'\nDeletion methods used:')
for d in report_1['deletion_execution']:
    print(f'  {d["data_id"]}: {d["deletion_method"]}')

print(f'\nModels requiring retraining: {len(report_1["retraining_required"])}')
for m in report_1['retraining_required']:
    print(f'  {m["model_name"]} — Priority: {m["priority"]}')


In [ ]:
# Scenario 2: Patient with fewer records
print('\n' + '='*80)
print('DELETION REQUEST #2: SUBJ-10002 (Patient with limited data)')
print('='*80)

report_2 = process_deletion_request('SUBJ-10002', data_inventory, model_lineage)
print(f'Status: {report_2["status"]}')
print(f'Records: {len(report_2["data_found"])}, Size: {report_2["total_data_size_mb"]:.1f} MB')

# Scenario 3: Non-existent subject
print('\n' + '='*80)
print('DELETION REQUEST #3: SUBJ-99999 (Non-existent subject)')
print('='*80)

report_3 = process_deletion_request('SUBJ-99999', data_inventory, model_lineage)
print(f'Status: {report_3["status"]}')


## Step 8: Model Lineage Impact Report


In [ ]:
print('MODEL-DATA IMPACT MATRIX (Request #1)')
print('='*80)

if isinstance(report_1['affected_models'], pd.DataFrame) and len(report_1['affected_models']) > 0:
    display_cols = ['model_name', 'impact_level', 'status', 'overlap_data_types', 
                    'retraining_required', 'priority', 'days_since_retrain']
    print(report_1['affected_models'][display_cols].to_string(index=False))
    
    total = len(report_1['affected_models'])
    need_retrain = (report_1['affected_models']['retraining_required'] == 'YES').sum()
    high_priority = (report_1['affected_models']['priority'] == 'HIGH').sum()
    print(f'\nSummary: {total} models affected, {need_retrain} need retraining, {high_priority} are HIGH priority')
else:
    print('No model impact data available — check your assess_model_impact implementation')


## Step 9: Audit Trail


In [ ]:
print(f'AUDIT TRAIL: {report_1["request_id"]}')
print('='*80)

for entry in report_1['audit_entries']:
    ts = entry['timestamp'].strftime('%Y-%m-%d %H:%M:%S')
    print(f'\n[{ts}] {entry["event"]}')
    print(f'  Status: {entry["status"]}')
    print(f'  Details: {entry["details"]}')


## Step 10: Visualizations (L6-R004: TODO)

Compliance reports get a lot more usable when the audit-trail data is plotted, not just listed. Build two simple visualizations from the data your pipeline already produced.


In [ ]:
# TODO Step 10: Add two visualizations from your processed deletion reports.
#
# Visualization 1 — Deletion method distribution (horizontal bar chart):
#   - X-axis: count of records deleted using each method.
#   - Y-axis: deletion method name (secure_erase, cryptographic_erasure,
#     standard_delete, archival).
#   - Data source: report_1['deletion_execution'] (list of dicts with
#     'deletion_method' key) — pandas Series.value_counts() makes this one line.
#   - Suggested title: "Deletion Methods Used — Request {request_id}".
#
# Visualization 2 — Model-data impact matrix (heatmap):
#   - Rows: affected model names (from report_1['affected_models']).
#   - Columns: deleted data types (from report_1['data_found']).
#   - Color: impact level (red for critical/high, yellow for medium, green for low).
#   - Build a small DataFrame with rows=models, cols=data_types, values=impact_level
#     mapped to a numeric scale, then plot via matplotlib.imshow().
#
# Hint: matplotlib.pyplot is already imported as plt. The solution uses
# plt.subplots() for layout, ax.barh() for the bar chart, and plt.imshow() for
# the heatmap.

print('Visualization TODOs:')
print('  1. Deletion method distribution bar chart')
print('  2. Model-data impact matrix heatmap')


## Summary

This notebook implemented the core logic for **GDPR Article 17 + CPRA §1798.105** deletion processing:

1. **Deletion method selection** — Matching data sensitivity and storage type to the appropriate deletion technique (Step 2)
2. **Retention conflict detection** — Routing records under FDA / consent / audit holds away from automated deletion (Step 3, L6-R001)
3. **Large-record flagging** — Surfacing > 1 GB records that need backup-rotation coordination for manual review (Step 4, L6-R006)
4. **Model impact assessment** — Cross-referencing deleted data against model training dependencies to flag retraining needs (Step 5)
5. **End-to-end pipeline** — Processing deletion requests with full audit trails for compliance documentation (Steps 6–9)
6. **Reporting visualizations** — Deletion-method distribution and model-impact heatmap for the compliance report (Step 10, L6-R004)

**Key governance insight:** Deleting personal data in an AI system isn't just about removing records — it requires (a) checking regulatory exemptions BEFORE deleting, (b) assessing downstream model impacts to make informed retraining decisions, and (c) keeping the audit trail credible for both GDPR Art. 17 and CPRA §1798.105 verification.
